In [3]:
# build_train_spacy.py
import spacy
from spacy.tokens import DocBin

def find_all(text, sub):
    """Yield (start, end) for each non-overlapping occurrence of sub in text."""
    i = 0
    while True:
        i = text.find(sub, i)
        if i == -1:
            return
        yield (i, i + len(sub))
        i += len(sub)

def add_ents_by_substrings(doc, ents_spec):
    """
    ents_spec: list of (substring, label[, occurrence_index])
      - If occurrence_index is omitted, first occurrence is used.
      - If substring appears multiple times and you want the Nth, pass the index (0-based).
    Returns a list[Span] for doc. Skips any that can't be aligned.
    """
    spans = []
    for item in ents_spec:
        if len(item) == 2:
            sub, label = item
            occ_idx = 0
        else:
            sub, label, occ_idx = item

        occs = list(find_all(doc.text, sub))
        if not occs:
            print(f"[warn] substring not found: {sub!r} in: {doc.text}")
            continue
        if occ_idx >= len(occs):
            print(f"[warn] occurrence {occ_idx} out of range for {sub!r} in: {doc.text}")
            continue

        start, end = occs[occ_idx]
        span = doc.char_span(start, end, label=label, alignment_mode="contract")
        if span is None:
            # try expand as a fallback
            span = doc.char_span(start, end, label=label, alignment_mode="expand")
        if span is None:
            print(f"[warn] could not align span for {sub!r} in: {doc.text}")
            continue
        spans.append(span)
    return spans

# ---------- Define your dataset with substrings, not offsets ----------
examples = [
    ("Tokyo Tower is 333m tall.", [("Tokyo Tower", "BUILDING")]),
    ("Apple released the first iPhone in 2007.", [("Apple", "ORG"), ("iPhone", "PRODUCT"), ("2007", "DATE")]),
    ("Barack Obama was the 44th president of the United States.", [("Barack Obama", "PERSON"), ("United States", "GPE")]),
    ("Google announced a new office in London.", [("Google", "ORG"), ("London", "GPE")]),
    ("Amazon acquired Whole Foods in 2017.", [("Amazon", "ORG"), ("Whole Foods", "ORG"), ("2017", "DATE")]),
    ("The Eiffel Tower is located in Paris.", [("Eiffel Tower", "BUILDING"), ("Paris", "GPE")]),
    ("Microsoft was founded by Bill Gates and Paul Allen.", [("Microsoft", "ORG"), ("Bill Gates", "PERSON"), ("Paul Allen", "PERSON")]),
    ("Tesla opened a Gigafactory in Berlin.", [("Tesla", "ORG"), ("Gigafactory", "PRODUCT"), ("Berlin", "GPE")]),
    ("World War II ended in 1945.", [("World War II", "EVENT"), ("1945", "DATE")]),
    ("Mount Everest is the tallest mountain in the world.", [("Mount Everest", "LOC")]),
    ("Facebook rebranded itself as Meta in 2021.", [("Facebook", "ORG"), ("Meta", "ORG"), ("2021", "DATE")]),
    ("IBM introduced Watson at a tech conference in Las Vegas.", [("IBM", "ORG"), ("Watson", "PRODUCT"), ("Las Vegas", "GPE")]),
    ("New York City is known as the Big Apple.", [("New York City", "GPE"), ("Big Apple", "ORG")]),
    ("The Great Wall of China stretches thousands of miles.", [("Great Wall of China", "BUILDING")]),
    ("Jeff Bezos stepped down as Amazon CEO in 2021.", [("Jeff Bezos", "PERSON"), ("Amazon", "ORG"), ("2021", "DATE")]),
    ("Elon Musk founded SpaceX in 2002.", [("Elon Musk", "PERSON"), ("SpaceX", "ORG"), ("2002", "DATE")]),
    ("The Colosseum is a famous landmark in Rome.", [("Colosseum", "BUILDING"), ("Rome", "GPE")]),
    ("The iPad Pro is one of Apple's best-selling products.", [("iPad Pro", "PRODUCT"), ("Apple", "ORG")]),
    ("Samsung unveiled the Galaxy S21 in January 2021.", [("Samsung", "ORG"), ("Galaxy S21", "PRODUCT"), ("January 2021", "DATE")]),
    ("The United Nations headquarters is in New York.", [("United Nations", "ORG"), ("New York", "GPE")]),
    ("Harry Potter was first published in 1997.", [("Harry Potter", "WORK_OF_ART"), ("1997", "DATE")]),
    ("The Golden Gate Bridge is located in San Francisco.", [("Golden Gate Bridge", "BUILDING"), ("San Francisco", "GPE")]),
    ("LinkedIn was acquired by Microsoft in 2016.", [("LinkedIn", "ORG"), ("Microsoft", "ORG"), ("2016", "DATE")]),
    ("The Statue of Liberty was a gift from France.", [("Statue of Liberty", "BUILDING"), ("France", "GPE")]),
    ("The COVID-19 pandemic began in 2019.", [("COVID-19", "EVENT"), ("2019", "DATE")]),
    ("Steve Jobs co-founded Apple in 1976.", [("Steve Jobs", "PERSON"), ("Apple", "ORG"), ("1976", "DATE")]),
    ("The Sydney Opera House is a UNESCO World Heritage Site.", [("Sydney Opera House", "BUILDING")]),
    ("NVIDIA announced Blackwell at GTC 2024.", [("NVIDIA", "ORG"), ("Blackwell", "PRODUCT"), ("GTC", "EVENT"), ("2024", "DATE")]),
    ("OpenAI released GPT-4 in March 2023.", [("OpenAI", "ORG"), ("GPT-4", "PRODUCT"), ("March 2023", "DATE")]),
    ("The Louvre is in Paris, France.", [("Louvre", "BUILDING"), ("Paris", "GPE"), ("France", "GPE")]),
    ("Alibaba operates a large marketplace in Hangzhou.", [("Alibaba", "ORG"), ("Hangzhou", "GPE")]),
    ("The Burj Khalifa stands 828 meters tall in Dubai.", [("Burj Khalifa", "BUILDING"), ("Dubai", "GPE")]),
    ("Stanford University is located in California.", [("Stanford University", "ORG"), ("California", "GPE")]),
    ("Twitter rebranded to X in 2023.", [("Twitter", "ORG"), ("X", "ORG"), ("2023", "DATE")]),
    ("The FIFA World Cup was hosted by Qatar in 2022.", [("FIFA World Cup", "EVENT"), ("Qatar", "GPE"), ("2022", "DATE")]),
]

# ---------- Build DocBin ----------
nlp = spacy.blank("en")
db = DocBin(store_user_data=True)  # store_user_data optional but handy

for text, ents_spec in examples:
    # Normalize smart quotes to avoid substring index surprises
    norm_text = text.replace("’", "'").replace("“", '"').replace("”", '"')
    doc = nlp(norm_text)
    spans = add_ents_by_substrings(doc, ents_spec)
    doc.ents = spans
    db.add(doc)

db.to_disk("train.spacy")
print("Wrote train.spacy with", len(examples), "examples.")


Wrote train.spacy with 35 examples.


In [4]:
# build_dev_spacy.py
import spacy
from spacy.tokens import DocBin

def find_all(text, sub):
    i = 0
    while True:
        i = text.find(sub, i)
        if i == -1:
            return
        yield (i, i + len(sub))
        i += len(sub)

def add_ents_by_substrings(doc, ents_spec):
    spans = []
    for item in ents_spec:
        sub, label, *rest = item
        occ_idx = rest[0] if rest else 0
        occs = list(find_all(doc.text, sub))
        if not occs or occ_idx >= len(occs):
            continue
        start, end = occs[occ_idx]
        span = doc.char_span(start, end, label=label, alignment_mode="contract") \
               or doc.char_span(start, end, label=label, alignment_mode="expand")
        if span:
            spans.append(span)
    return spans

dev_examples = [
    ("Apple opened a new store in Chicago.", [("Apple","ORG"), ("Chicago","GPE")]),
    ("The Empire State Building is in New York.", [("Empire State Building","BUILDING"), ("New York","GPE")]),
    ("NVIDIA showcased Blackwell at GTC 2024.", [("NVIDIA","ORG"), ("Blackwell","PRODUCT"), ("GTC","EVENT"), ("2024","DATE")]),
    ("Barack Obama was born in 1961.", [("Barack Obama","PERSON"), ("1961","DATE")]),
    ("The Louvre Museum is in Paris, France.", [("Louvre","BUILDING"), ("Paris","GPE"), ("France","GPE")]),
    ("Microsoft acquired GitHub in 2018.", [("Microsoft","ORG"), ("GitHub","ORG"), ("2018","DATE")]),
]

nlp = spacy.blank("en")
db = DocBin(store_user_data=True)
for text, ents_spec in dev_examples:
    text = text.replace("’","'").replace("“",'"').replace("”",'"')
    doc = nlp(text)
    doc.ents = add_ents_by_substrings(doc, ents_spec)
    db.add(doc)

db.to_disk("dev.spacy")
print("Wrote dev.spacy with", len(dev_examples), "examples.")


Wrote dev.spacy with 6 examples.
